# 진짜 최종 구현 코드


1. 문제의 본질
원래 amount는 
- offer completed 시점에 붙어있던 값임
- 근데 이 값이 항상 위 조건을 따르진 않았음

2. 1차 해결 로직 : completion_amount_current
offer received를 시작 시간으로 보고 offer completed가 되는 시간까지 사이의 amount를 더한 값

3. 2차 해결 로직 : completion_amount_prev_received
1차 해결 로직으로 구할 수 없는 경우(76건)
- 같은 프로모션(offer)가 한 번 더 온 경우
- received(1) -> amount -> received(2) -> viewed -> completed = amount의 경우
    1차 로직으로는 received(2)부터 completed 까지의 amount를 더함.
    이렇게 되면 만약 received(1)에 대한 completed일 경우 received(1)와 received(2) 사이 발생한 amount를 제외시키게됨 -> difficulty 미충족
- 이를 해결하기 위해 received(2) 이전의 received(1)을 기준으로 계산하는 로직 만듬

4. 최종 df는?
2가지 로직에 충족한 amount를 맞게 적용
- 기본은 completion_amount_current으로
- 부족한 경우 completion_amount_prev_received를 적용


In [1]:
# # 76건만 다시 뽑기
# anomaly = promo_seq[
#     (promo_seq['is_completed'].eq(1)) &
#     (promo_seq['is_normal_flow'].eq(1)) &
#     (promo_seq['completion_amount_current'].lt(promo_seq['difficulty']))
# ].copy()

# # 현재 기준 / 이전 received 기준 amount 같이 보기
# anomaly['promo_effect_current'] = anomaly['completion_amount_current']
# anomaly['promo_effect_prev_received'] = anomaly['completion_amount_prev_received']

# display(
#     anomaly[
#         [
#             'person', 'offer_id', 'offer_cycle',
#             'prev_received', 'offer received', 'offer viewed', 'offer completed',
#             'difficulty',
#             'completion_amount_current', 'completion_amount_prev_received',
#             'promo_effect_current', 'promo_effect_prev_received'
#         ]
#     ].sort_values(['person', 'offer_id', 'offer received'])
# )


In [2]:
# # 76건 각각에 대해 received 전후 타임라인 확인
# for _, row in anomaly.sort_values(['person', 'offer_id', 'offer received']).iterrows():
#     person = row['person']
#     offer_id = row['offer_id']
#     start = row['prev_received'] if pd.notna(row['prev_received']) else row['offer received']
#     end = row['offer completed']


#     print(f"\n### person={person}, offer_id={offer_id}, offer_cycle={row['offer_cycle']}")
#     print(f"current received: {row['offer received']}, prev received: {row['prev_received']}, completed: {row['offer completed']}")

#     window = merged[
#         (merged['person'] == person) &
#         (merged['time'] >= start) &
#         (merged['time'] <= end)
#     ].sort_values(['time', 'Unnamed: 0'])

#     display(window[['time', 'event', 'amount', 'offer_id', 'offer_type', 'difficulty', 'duration']])


In [3]:
import numpy as np
import pandas as pd

promotion_df = pd.read_csv('./data/promotion_df.csv')
merged_df = pd.read_csv('./data/all_merged.csv')


In [4]:
transactions = merged_df[merged_df['event'] == 'transaction'].copy()
transactions = transactions[['person', 'time', 'amount']]
transactions = transactions.sort_values(['person', 'time']).reset_index(drop=True)

views = merged_df[merged_df['event'] == 'offer viewed'].copy()
views = views[['person', 'offer_id', 'time']]
views = views.sort_values(['person', 'offer_id', 'time']).reset_index(drop=True)

tx_by_person = {}
for _, row in transactions.iterrows():
    person = row['person']
    if person not in tx_by_person:
        tx_by_person[person] = []
    tx_by_person[person].append((row['time'], row['amount']))

view_by_person_offer = {}
for _, row in views.iterrows():
    key = (row['person'], row['offer_id'])
    if key not in view_by_person_offer:
        view_by_person_offer[key] = []
    view_by_person_offer[key].append(row['time'])


In [5]:
promotion_df = promotion_df.sort_values(
    ['person', 'offer_id', 'offer received', 'offer completed']
).reset_index(drop=True)


In [6]:
completion_amount_current_list = []

for _, row in promotion_df.iterrows():
    person = row['person']
    start_time = row['offer received']
    end_time = row['offer completed']

    total = 0.0
    if person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= start_time and tx_time <= end_time:
                total += amount

    completion_amount_current_list.append(round(total, 2))

promotion_df['completion_amount_current'] = completion_amount_current_list

In [7]:
transactions = merged_df[merged_df['event'] == 'transaction'].copy()
transactions = transactions[['person', 'time', 'amount']]
transactions = transactions.sort_values(['person', 'time']).reset_index(drop=True)

views = merged_df[merged_df['event'] == 'offer viewed'].copy()
views = views[['person', 'offer_id', 'time']]
views = views.sort_values(['person', 'offer_id', 'time']).reset_index(drop=True)

tx_by_person = {}
for _, row in transactions.iterrows():
    person = row['person']
    if person not in tx_by_person:
        tx_by_person[person] = []
    tx_by_person[person].append((row['time'], row['amount']))

view_by_person_offer = {}
for _, row in views.iterrows():
    key = (row['person'], row['offer_id'])
    if key not in view_by_person_offer:
        view_by_person_offer[key] = []
    view_by_person_offer[key].append(row['time'])


In [8]:
promotion_df = promotion_df.sort_values(
    ['person', 'offer_id', 'offer received', 'offer completed']
).reset_index(drop=True)


In [9]:
completion_amount_current_list = []

for _, row in promotion_df.iterrows():
    person = row['person']
    start_time = row['offer received']
    end_time = row['offer completed']

    total = 0.0
    if person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= start_time and tx_time <= end_time:
                total += amount

    completion_amount_current_list.append(round(total, 2))

promotion_df['completion_amount_current'] = completion_amount_current_list


In [10]:
promo_influenced_amount_list = []

for _, row in promotion_df.iterrows():
    person = row['person']
    viewed_time = row['offer viewed']
    end_time = row['offer completed']

    if pd.isna(viewed_time):
        promo_influenced_amount_list.append(np.nan)
        continue

    total = 0.0
    if person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= viewed_time and tx_time <= end_time:
                total += amount

    promo_influenced_amount_list.append(round(total, 2))

promotion_df['promo_influenced_amount'] = promo_influenced_amount_list


In [11]:
anomaly = promotion_df[
    (promotion_df['is_completed'].eq(1)) &
    (promotion_df['is_normal_flow'].eq(1)) &
    (promotion_df['completion_amount_current'] < promotion_df['difficulty'])
].copy()

print('anomaly 행 수:', len(anomaly))


anomaly 행 수: 76


In [12]:
prev_helper = promotion_df[
    ['person', 'offer_id', 'offer_cycle', 'offer received']
].copy()

prev_helper = prev_helper.sort_values(
    ['person', 'offer_id', 'offer received']
).reset_index(drop=True)

prev_helper['prev_received'] = prev_helper.groupby(['person', 'offer_id'])['offer received'].shift(1)

anomaly = anomaly.drop(columns=['prev_received'], errors='ignore')

anomaly = anomaly.merge(
    prev_helper[['person', 'offer_id', 'offer_cycle', 'prev_received']],
    on=['person', 'offer_id', 'offer_cycle'],
    how='left'
)


In [13]:
completion_amount_prev_received_list = []

for _, row in anomaly.iterrows():
    person = row['person']
    start_time = row['prev_received']
    end_time = row['offer completed']

    if pd.isna(start_time):
        completion_amount_prev_received_list.append(np.nan)
        continue

    total = 0.0
    if person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= start_time and tx_time <= end_time:
                total += amount

    completion_amount_prev_received_list.append(round(total, 2))

anomaly['completion_amount_prev_received'] = completion_amount_prev_received_list


In [14]:
prev_viewed_list = []
promo_influenced_amount_prev_received_list = []

for _, row in anomaly.iterrows():
    person = row['person']
    offer_id = row['offer_id']
    prev_received = row['prev_received']
    end_time = row['offer completed']

    prev_viewed = np.nan
    total = 0.0

    if pd.notna(prev_received):
        key = (person, offer_id)
        if key in view_by_person_offer:
            for view_time in view_by_person_offer[key]:
                if view_time >= prev_received and view_time <= end_time:
                    prev_viewed = view_time
                    break

        if pd.notna(prev_viewed) and person in tx_by_person:
            for tx_time, amount in tx_by_person[person]:
                if tx_time >= prev_viewed and tx_time <= end_time:
                    total += amount

    prev_viewed_list.append(prev_viewed)
    if pd.isna(prev_viewed):
        promo_influenced_amount_prev_received_list.append(np.nan)
    else:
        promo_influenced_amount_prev_received_list.append(round(total, 2))

anomaly['prev_viewed'] = prev_viewed_list
anomaly['promo_influenced_amount_prev_received'] = promo_influenced_amount_prev_received_list


In [15]:
promotion_df = promotion_df.drop(columns=['completion_amount_prev_received'], errors='ignore')

promotion_df = promotion_df.merge(
    anomaly[
        [
            'person', 'offer_id', 'offer_cycle',
            'prev_received',
            'completion_amount_prev_received',
            'prev_viewed',
            'promo_influenced_amount_prev_received'
        ]
    ],
    on=['person', 'offer_id', 'offer_cycle'],
    how='left'
)


In [16]:
promotion_df['adjusted_completion_amount'] = promotion_df['completion_amount_current']

fix_mask = promotion_df['completion_amount_prev_received'].ge(promotion_df['difficulty'])
promotion_df.loc[fix_mask, 'adjusted_completion_amount'] = promotion_df.loc[fix_mask, 'completion_amount_prev_received']


In [17]:
display(
    promotion_df[
        [
            'person', 'offer_id', 'offer_cycle',
            'offer received', 'prev_received',
            'offer viewed', 'prev_viewed',
            'offer completed', 'difficulty',
            'completion_amount_current',
            'completion_amount_prev_received',
            'promo_influenced_amount',
            'promo_influenced_amount_prev_received',
            'adjusted_completion_amount'
        ]
    ].head(20)
)


,person,offer_id,offer_cycle,offer received,prev_received,offer viewed,prev_viewed,offer completed,difficulty,completion_amount_current,completion_amount_prev_received,promo_influenced_amount,promo_influenced_amount_prev_received,adjusted_completion_amount
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,NaN,456.0,NaN,414.0,5,8.57,NaN,0.00,NaN,8.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,NaN,540.0,NaN,528.0,10,14.11,NaN,0.00,NaN,14.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,NaN,NaN,576.0,10,10.27,NaN,NaN,NaN,10.27
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,NaN,192.0,NaN,NaN,0,0.00,NaN,0.00,NaN,0.00
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,NaN,372.0,NaN,NaN,0,0.00,NaN,0.00,NaN,0.00
5,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,NaN,216.0,NaN,NaN,5,0.00,NaN,0.00,NaN,0.00
6,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,NaN,630.0,NaN,NaN,5,0.00,NaN,0.00,NaN,0.00
7,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,NaN,516.0,NaN,576.0,5,22.05,NaN,22.05,NaN,22.05
8,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,NaN,432.0,NaN,576.0,20,22.05,NaN,22.05,NaN,22.05
9,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,NaN,186.0,NaN,252.0,7,11.93,NaN,11.93,NaN,11.93


In [18]:
remain = promotion_df[
    (promotion_df['is_completed'].eq(1)) &
    (promotion_df['is_normal_flow'].eq(1)) &
    (promotion_df['adjusted_completion_amount'] < promotion_df['difficulty'])
].copy()

print('여전히 difficulty보다 작은 정상 completed 건수:', len(remain))


여전히 difficulty보다 작은 정상 completed 건수: 0


In [19]:
promotion_final = promotion_df.copy()

# 원래 amount는 보관
promotion_final['amount_raw'] = promotion_final['amount']

# 최종 amount는 보정값으로 교체
promotion_final['amount'] = promotion_final['adjusted_completion_amount']


In [20]:
# promo 영향액도 최종값으로 정리
promotion_final['promo_influenced_amount_final'] = promotion_final['promo_influenced_amount']

mask = promotion_final['completion_amount_current'] < promotion_final['difficulty']
promotion_final.loc[mask, 'promo_influenced_amount_final'] = \
    promotion_final.loc[mask, 'promo_influenced_amount_prev_received']


In [21]:
promotion_final.head(20)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,is_deduplicated,completion_amount_current,promo_influenced_amount,prev_received,completion_amount_prev_received,prev_viewed,promo_influenced_amount_prev_received,adjusted_completion_amount,amount_raw,promo_influenced_amount_final
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,0,8.57,0.00,NaN,NaN,NaN,NaN,8.57,8.57,0.00
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,0,14.11,0.00,NaN,NaN,NaN,NaN,14.11,14.11,0.00
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,0,10.27,NaN,NaN,NaN,NaN,NaN,10.27,10.27,NaN
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,informational,0,0,3,...,0,0.00,0.00,NaN,NaN,NaN,NaN,0.00,NaN,0.00
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,informational,0,0,4,...,0,0.00,0.00,NaN,NaN,NaN,NaN,0.00,NaN,0.00
5,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,bogo,5,5,5,...,0,0.00,0.00,NaN,NaN,NaN,NaN,0.00,NaN,NaN
6,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,bogo,5,5,5,...,0,0.00,0.00,NaN,NaN,NaN,NaN,0.00,NaN,NaN
7,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,1,22.05,22.05,NaN,NaN,NaN,NaN,22.05,22.05,22.05
8,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,discount,20,5,10,...,0,22.05,22.05,NaN,NaN,NaN,NaN,22.05,22.05,22.05
9,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,1,11.93,11.93,NaN,NaN,NaN,NaN,11.93,11.93,11.93


In [ ]:
# # 저장
# promotion_final.to_csv('./data/promotion_final.csv', index=False)


### 정상흐름 amount 구해보기

In [25]:
normal_total_before = promotion_final.loc[
    promotion_final['is_deduplicated'].eq(1),
    'amount_raw'
].sum()

normal_total_after = promotion_final.loc[
    promotion_final['is_deduplicated'].eq(1),
    'amount'
].sum()

print('수정 전 정상흐름 amount 합:', normal_total_before)
print('수정 후 정상흐름 amount 합:', normal_total_after)
print('두 amount 차이:', round(normal_total_after - normal_total_before, 2))


수정 전 정상흐름 amount 합: 442993.24
수정 후 정상흐름 amount 합: 486645.59
두 amount 차이: 43652.35


# `amount` 재계산 및 보정 정리

이 노트북에서는 `promotion_df.csv`를 기준으로 offer row의 `amount`를 다시 계산했다.

## 작업 내용
- `offer received`부터 `offer completed`까지의 transaction 금액을 다시 합쳐 `completion_amount_current`를 계산했다.
- 현재 received 기준 금액이 `difficulty`를 못 넘는 정상 흐름 행만 따로 뽑았다.
- 그 행들에 한해서만 이전 `received`를 시작점으로 다시 계산한 `completion_amount_prev_received`를 확인했다.
- `current`가 부족할 때만 `prev_received`를 보정값으로 사용했다.
- `offer viewed` 이후의 프로모션 영향액도 `promo_influenced_amount`와 `promo_influenced_amount_prev_received`로 함께 계산했다.
- 최종적으로 `adjusted_completion_amount`를 만들고, `amount_raw`는 원본값으로 보존했다.
- 저장용 최종 파일은 `promotion_final.csv`로 만들었다.

## 핵심 정리
- `completion_amount_current` = 현재 received 기준 총 결제액
- `completion_amount_prev_received` = 이전 received 기준 총 결제액
- `promo_influenced_amount` = viewed 이후 결제액
- `promo_influenced_amount_prev_received` = 이전 received 기준 viewed 이후 결제액
- `adjusted_completion_amount` = 최종 보정된 amount

## 해석
- `current`가 충분하면 그대로 사용했다.
- `current`가 부족한 행만 `prev_received`로 보정했다.
- 정상 흐름의 offer row를 더 정확하게 해석할 수 있도록 amount를 다시 정리한 작업이다.
